In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
candidate_roots = [Path.cwd(), *Path.cwd().parents]
reports_root = None
for root in candidate_roots:
    candidate = root / "reports" / "msm_funding_v0"
    if candidate.exists():
        reports_root = candidate
        break

if reports_root is None:
    raise FileNotFoundError("Could not locate reports/msm_funding_v0 from current working directory")

run_dirs = [p for p in reports_root.iterdir() if p.is_dir()]
if not run_dirs:
    raise FileNotFoundError(f"No run directories found in {reports_root}")

latest_run_dir = max(run_dirs, key=lambda p: p.stat().st_mtime)
csv_path = latest_run_dir / "msm_timeseries.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"Expected file not found: {csv_path}")

df = pd.read_csv(csv_path)
date_col = "date" if "date" in df.columns else "decision_date" if "decision_date" in df.columns else None
if date_col is None:
    raise KeyError("Neither 'date' nor 'decision_date' found in msm_timeseries.csv")

df[date_col] = pd.to_datetime(df[date_col], utc=True, errors="raise")

display(latest_run_dir)
df.head()

In [ ]:
if "F_tk_apr" not in df.columns:
    raise KeyError("Column 'F_tk_apr' not found in dataset")

state_history = []
is_on = False

for f_apr_dec in df["F_tk_apr"]:
    # Entry gate
    if (not is_on) and (0.01 <= f_apr_dec <= 0.04):
        is_on = True
    # Exit gate (cold)
    elif is_on and (f_apr_dec < 0.0):
        is_on = False
    # Exit gate (toxic)
    elif is_on and (f_apr_dec > 0.04):
        is_on = False

    state_history.append(is_on)

df["portfolio_state"] = state_history
df[[date_col, "F_tk_apr", "portfolio_state"]].tail()

In [ ]:
total_epochs = len(df)
if total_epochs == 0:
    raise ValueError("Dataset is empty; cannot compute regime summary")

on_count = int(df["portfolio_state"].sum())
off_count = total_epochs - on_count
on_pct = (on_count / total_epochs) * 100.0
off_pct = (off_count / total_epochs) * 100.0

turnover = int(df["portfolio_state"].ne(df["portfolio_state"].shift(1)).sum() - 1)
turnover = max(turnover, 0)

last_row = df.iloc[-1]
live_date = last_row[date_col].isoformat()
live_f_apr = float(last_row["F_tk_apr"])
live_state = bool(last_row["portfolio_state"])

print("=== ASYMMETRIC SAFE-STICKY REGIME AUDIT ===")
print(f"Master N-Count: {total_epochs:,} epochs")
print(
    "Historical Allocation: "
    f"ON={on_pct:,.2f}% ({on_count:,} epochs) | "
    f"OFF={off_pct:,.2f}% ({off_count:,} epochs)"
)
print(f"Turnover: {turnover:,} regime shifts")
print("LIVE SENSOR:")
print(
    f"- date={live_date} | "
    f"F_tk_apr={live_f_apr:.2%} | "
    f"portfolio_state={'ON' if live_state else 'OFF'}"
)

In [ ]:
# Outlier Inquisition: diagnose whether -202.56% APR is broad vs single-asset

silver_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "data" / "curated" / "data_lake" / "silver_fact_funding.parquet"
    if candidate.exists():
        silver_path = candidate
        break

if silver_path is None:
    raise FileNotFoundError("Could not locate silver_fact_funding.parquet under data/curated/data_lake")

silver = pd.read_parquet(silver_path)

required_cols = {"asset_id", "funding_rate_raw_pct", "date"}
missing = required_cols - set(silver.columns)
if missing:
    raise KeyError(f"silver_fact_funding.parquet missing columns: {sorted(missing)}")

# Enforce strict UTC alignment for the epoch.
silver["date_utc"] = pd.to_datetime(silver["date"], utc=True, errors="raise")
target_epoch = pd.Timestamp("2026-03-24T00:00:00+00:00")

df_ts = silver[silver["date_utc"] == target_epoch].copy()
if df_ts.empty:
    raise ValueError(f"No rows found for target epoch: {target_epoch.isoformat()} in silver_fact_funding")

# If an asset_id appears multiple times (e.g., across exchanges/instruments), aggregate to an asset-level value.
asset_level = (
    df_ts.groupby("asset_id", as_index=False)["funding_rate_raw_pct"]
    .mean()
    .rename(columns={"funding_rate_raw_pct": "funding_rate_raw_pct"})
)

asset_level = asset_level.sort_values("funding_rate_raw_pct", ascending=True)

basket_mean = float(asset_level["funding_rate_raw_pct"].mean())
basket_median = float(asset_level["funding_rate_raw_pct"].median())

top10 = asset_level.head(10).copy()

format_pct = lambda x: f"{float(x):.2%}"

print("=== OUTLIER INQUISITION: -202.56% APR epoch ===")
print(f"EPOCH (UTC): {target_epoch.isoformat()}")
print(f"Basket mean funding_rate_raw_pct: {format_pct(basket_mean)}")
print(f"Basket median funding_rate_raw_pct: {format_pct(basket_median)}")
print("\nTop 10 most negative assets (by funding_rate_raw_pct):")

out = top10[["asset_id", "funding_rate_raw_pct"]].copy()
out["funding_rate_raw_pct"] = out["funding_rate_raw_pct"].map(format_pct)
print(out.to_string(index=False))

In [ ]:
# Slice the macro sensor table from Jan 2026 through 24 Mar 2026 (UTC, inclusive)

start_utc = pd.Timestamp("2026-01-01T00:00:00+00:00")
end_utc = pd.Timestamp("2026-03-24T00:00:00+00:00")

mask = (df[date_col] >= start_utc) & (df[date_col] <= end_utc)
df_window = df.loc[mask, [date_col, "F_tk", "F_tk_apr"]].copy()

# Add a human-readable percent column (matching the screenshot intent)
df_window["F_tk_apr_pct"] = (df_window["F_tk_apr"] * 100.0).round(2).map(lambda x: f"{x:.2f}%")

# Show the table
_df_show = df_window.rename(columns={date_col: "decision_date"})
display(_df_show)

# Export for PM handoff
out_dir = Path.cwd() / "out"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "msm_timeseries_2026-01-01_to_2026-03-24.csv"
_df_show.to_csv(out_path, index=False)
print(f"Saved: {out_path}")